# Phase 4: Generative Method — DiffPIR

## Theory

**DiffPIR** (Zhu et al., 2023 — *Denoising Diffusion Models for Plug-and-Play Image Restoration*)
combines a pre-trained **DDPM** denoiser with a classical data-consistency step to solve
the inverse problem:

$$\min_x \frac{1}{2}\|Kx - y^\delta\|^2_2 + \lambda \log p(x)$$

where $K$ is the (avg-pool) downscaling operator, $y^\delta$ is the noisy low-resolution
observation, and $\log p(x)$ is the prior encoded by the diffusion model.

### Algorithm (DiffPIR = HQS + DDIM)
At each DDIM step $t$ (reversed from $T$ to $0$):

| Step | Operation | Formula |
|------|-----------|----------|
| 1 | **Denoise** | $\hat{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t}\,\varepsilon_\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}}$ |
| 2 | **HQS proximal** (Fourier) | $\hat{x}_{\text{prox}} = \mathcal{F}^{-1}\!\left\{\frac{\mathcal{H}^*\mathcal{F}(K^\top y) + \rho\,\mathcal{F}(\hat{x}_0)}{|\mathcal{H}|^2 + \rho}\right\}$ |
| 3 | **DDIM update** | $x_{t-1} = \sqrt{\bar{\alpha}_{t-1}}\,\hat{x}_{\text{prox}} + \sqrt{1-\bar{\alpha}_{t-1}}\,\varepsilon_\theta$ |

where $\mathcal{H}$ is the Fourier response of the avg-pool PSF and
$\rho_t = \kappa\,\sigma_n^2 / (1-\bar{\alpha}_t)$.

### Comparison with DPS (homework 3)
| Feature | DPS (HM3) | **DiffPIR (NB04)** |
|---------|-----------|--------------------|
| Data consistency | gradient guidance | **closed-form proximal** |
| Sampling | DDPM (1000 steps) | **DDIM (100 steps, 10× faster)** |
| Complexity | O(N) grad per step | O(N log N) FFT per step |


## 4.1  Environment Setup (Colab / Local)

In [ ]:
import os, sys, shutil

# ── Detect runtime ──────────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_DRIVE_FOLDER = 'CI_Project'

if IN_COLAB:
    print('Running on Google Colab 🚀')
    drive.mount('/content/drive')
    DRIVE_PATH = os.path.join('/content/drive/MyDrive', BASE_DRIVE_FOLDER)
else:
    print('Running locally 💻')
    DRIVE_PATH = os.path.join('.', BASE_DRIVE_FOLDER)

DATASET_PATH = os.path.join(DRIVE_PATH, 'dataset')
WEIGHTS_PATH = os.path.join(DRIVE_PATH, 'weights')
OUTPUT_PATH  = os.path.join(DRIVE_PATH, 'outputs', 'diffpir')

os.makedirs(WEIGHTS_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH,  exist_ok=True)

print(f'Dataset : {DATASET_PATH}')
print(f'Weights : {WEIGHTS_PATH}')
print(f'Output  : {OUTPUT_PATH}')


## 4.2  Clone / Update IPPy

In [ ]:
if not os.path.exists('IPPy'):
    !git clone https://github.com/alessandrocampedelli/IPPy.git
else:
    %cd IPPy
    !git pull
    %cd ..

if 'IPPy' not in sys.path:
    sys.path.insert(0, 'IPPy')

from IPPy import operators
from IPPy.utilities import metrics
from IPPy.utilities._utilities import gaussian_noise, get_device
from IPPy.nn.diffusion import (
    DiffusionUNet, EMA, cosine_beta_schedule, extract, denormalize_to_01
)
print('✅ IPPy imported successfully')


## 4.3  Imports & Device

In [ ]:
import math, time, glob, copy
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DEVICE = get_device()
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
print(f'Device: {DEVICE}')


## 4.4  Hyperparameters

> **Training note:** with ~3 000 training images on a Colab T4 GPU, each epoch takes
> roughly **2–3 minutes**. 100 epochs ≈ 3–5 h. Training is checkpointed every 5 epochs
> so you can resume across sessions. If the weights file already exists the training
> cell is skipped automatically.
>
> **Inference note:** `NUM_TEST_IMAGES=20` images per scenario with `DDIM_STEPS=100`
> takes ≈ 5–10 min on T4. Increase `NUM_TEST_IMAGES` for a more thorough evaluation.

In [ ]:
# ── Degradation scenarios (same as NB02 & NB03) ─────────────────────────
ALL_SCENARIOS = [
    (2, 0.005),   # scale=2, noise=0.005
    (2, 0.01),    # scale=2, noise=0.01
    (4, 0.005),   # scale=4, noise=0.005
    (4, 0.01),    # scale=4, noise=0.01
]
IMG_SHAPE = (256, 256)

# ── Diffusion model architecture ─────────────────────────────────────────
BASE_CH       = 32             # base channels (small model for feasible training)
CHANNEL_MULTS = (1, 2, 4)     # → 32, 64, 128 channels across levels
TIME_DIM      = 128
DROPOUT       = 0.05
ATTN_LEVELS   = (1, 2)        # attention at deeper levels (saves memory at 256×256)

# ── Training ─────────────────────────────────────────────────────────────
T_DIFFUSION    = 1000          # total diffusion timesteps
N_TRAIN_EPOCHS = 100           # training epochs
BATCH_SIZE     = 8
LR_DIFFUSION   = 2e-4
EMA_DECAY      = 0.9999
CKPT_EVERY     = 5             # checkpoint every N epochs

# ── DiffPIR inference ────────────────────────────────────────────────────
DDIM_STEPS      = 100          # number of DDIM sampling steps (100 ≪ 1000)
ETA             = 0.0          # 0 = deterministic DDIM; 1 = full DDPM noise
RHO_SCALE       = 1.0          # HQS regularization scale κ (tune if needed)
NUM_TEST_IMAGES = 20           # test images per scenario (increase for final eval)

# ── File names ────────────────────────────────────────────────────────────
DIFF_WEIGHTS_FILE = os.path.join(WEIGHTS_PATH, 'DDPMDenoiser_retinal.pth')
DIFF_CKPT_FILE    = os.path.join(WEIGHTS_PATH, 'DDPMDenoiser_retinal_ckpt.pth')

print('Hyperparameters set ✅')
print(f'  Model: base_ch={BASE_CH}, mults={CHANNEL_MULTS}')
print(f'  Training: {N_TRAIN_EPOCHS} epochs, bs={BATCH_SIZE}, lr={LR_DIFFUSION}')
print(f'  Inference: DDIM_STEPS={DDIM_STEPS}, RHO_SCALE={RHO_SCALE}, '
      f'NUM_TEST_IMAGES={NUM_TEST_IMAGES}')


## 4.5  Diffusion Model Architecture

We use `DiffusionUNet` from IPPy — a time-conditioned U-Net that predicts
the noise $\varepsilon$ added at step $t$ (standard DDPM $\varepsilon$-prediction).

The architecture has **3 resolution levels** (256 → 128 → 64) with
self-attention at the deeper levels, keeping the $O(N^2)$ attention cost manageable.

In [ ]:
diffusion_model = DiffusionUNet(
    in_ch        = 1,
    base_ch      = BASE_CH,
    channel_mults= CHANNEL_MULTS,
    time_dim     = TIME_DIM,
    dropout      = DROPOUT,
    attn_levels  = ATTN_LEVELS,
).to(DEVICE)

n_params = sum(p.numel() for p in diffusion_model.parameters() if p.requires_grad)
print(f'DiffusionUNet trainable parameters: {n_params/1e6:.2f} M')


## 4.6  Training Dataset

The diffusion model is trained **unconditionally** on HR images only
(no degradation). It learns $p(x)$ — the prior over clean retinal images.

All images are pre-loaded into RAM to avoid repeated Drive I/O.
Images are normalised to $[-1, 1]$ as required by the DDPM framework.

In [ ]:
class RetinalDataset(Dataset):
    """
    Loads all HR .pt tensors from data_dir into RAM, normalises to [-1, 1].
    """
    def __init__(self, data_dir: str):
        files = sorted(glob.glob(os.path.join(data_dir, '*.pt')))
        assert len(files) > 0, f'No .pt files found in {data_dir}'
        print(f'Caching {len(files)} tensors from {data_dir} ...')
        self.data = [torch.load(f, weights_only=True) for f in tqdm(files, leave=False)]
        ram_mb = sum(d.element_size() * d.nelement() for d in self.data) / 1e6
        print(f'  ✅ Cached {len(self.data)} images ({ram_mb:.0f} MB in RAM)')

    def __len__(self):  return len(self.data)

    def __getitem__(self, idx):
        # Normalise from [0, 1] → [-1, 1]
        return 2.0 * self.data[idx] - 1.0


## 4.7  Diffusion Training

**Standard DDPM training** (Ho et al. 2020): at each step sample a random
timestep $t$, corrupt the image, and train the network to predict the noise.

$$\mathcal{L} = \mathbb{E}_{x_0, \varepsilon, t}\left[\|\varepsilon - \varepsilon_\theta(\sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon,\, t)\|^2\right]$$

**EMA** (Exponential Moving Average) is applied on model weights for stable inference.
Checkpoints are saved every `CKPT_EVERY` epochs → resume-safe across Colab sessions.

In [ ]:
# ── Build schedule ──────────────────────────────────────────────────────
betas_sched    = cosine_beta_schedule(T_DIFFUSION).to(DEVICE)
alphas_sched   = 1.0 - betas_sched
alpha_bars_sched = torch.cumprod(alphas_sched, dim=0)

if os.path.exists(DIFF_WEIGHTS_FILE):
    print(f'✅ Diffusion model already trained → {DIFF_WEIGHTS_FILE}')
    print('   Skipping training. Load weights in Section 4.9.')
else:
    # ── Data ──────────────────────────────────────────────────────────
    train_ds     = RetinalDataset(os.path.join(DATASET_PATH, 'train'))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=torch.cuda.is_available())

    # ── Optimizer & EMA ───────────────────────────────────────────────
    optimizer = torch.optim.AdamW(diffusion_model.parameters(),
                                   lr=LR_DIFFUSION, weight_decay=1e-4)
    ema = EMA(diffusion_model, decay=EMA_DECAY)

    # ── Resume from checkpoint ────────────────────────────────────────
    start_epoch = 1
    train_history = {'loss': []}
    if os.path.exists(DIFF_CKPT_FILE):
        ckpt = torch.load(DIFF_CKPT_FILE, map_location=DEVICE, weights_only=False)
        diffusion_model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        ema.shadow   = ckpt['ema_shadow']
        train_history = ckpt.get('history', {'loss': []})
        start_epoch  = ckpt['epoch'] + 1
        print(f'🔄 Resuming from epoch {start_epoch}/{N_TRAIN_EPOCHS}')

    # ── Training loop ─────────────────────────────────────────────────
    print(f'Starting DDPM training: {N_TRAIN_EPOCHS} epochs on {len(train_ds)} images')
    for epoch in range(start_epoch, N_TRAIN_EPOCHS + 1):
        diffusion_model.train()
        epoch_loss = 0.0
        t0 = time.time()

        for x_batch in tqdm(train_loader, desc=f'Ep {epoch}/{N_TRAIN_EPOCHS}',
                             leave=False):
            x_batch = x_batch.to(DEVICE)
            # Sample random diffusion timesteps
            t_rand = torch.randint(0, T_DIFFUSION, (x_batch.shape[0],),
                                   device=DEVICE, dtype=torch.long)
            # Forward diffusion: x_t = sqrt(ᾱ_t) x0 + sqrt(1-ᾱ_t) ε
            eps     = torch.randn_like(x_batch)
            ab_t    = extract(alpha_bars_sched, t_rand, x_batch.shape)
            x_t     = torch.sqrt(ab_t) * x_batch + torch.sqrt(1 - ab_t) * eps
            # Predict noise
            optimizer.zero_grad()
            eps_pred = diffusion_model(x_t, t_rand)
            loss     = F.mse_loss(eps_pred, eps)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), 1.0)
            optimizer.step()
            ema.update()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        train_history['loss'].append(avg_loss)
        elapsed  = time.time() - t0
        print(f'  [{epoch:3d}/{N_TRAIN_EPOCHS}]  loss={avg_loss:.5f}  ({elapsed:.0f}s)')

        # Checkpoint
        if epoch % CKPT_EVERY == 0 or epoch == N_TRAIN_EPOCHS:
            torch.save({
                'epoch'             : epoch,
                'model_state_dict'  : diffusion_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'ema_shadow'        : ema.shadow,
                'history'           : train_history,
            }, DIFF_CKPT_FILE)

    # Save final EMA weights
    ema.apply_shadow()
    torch.save(diffusion_model.state_dict(), DIFF_WEIGHTS_FILE)
    ema.restore()
    if os.path.exists(DIFF_CKPT_FILE):
        os.remove(DIFF_CKPT_FILE)
    print(f'\n✅ Training complete → {DIFF_WEIGHTS_FILE}')


## 4.8  Training History

In [ ]:
# Plot training loss (only if training was run in this session)
if 'train_history' in dir() and len(train_history['loss']) > 0:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(train_history['loss'], linewidth=1.5, color='steelblue')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.set_title('DDPM Training Loss — Retinal Dataset')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_PATH, 'diffpir_training_loss.png'), dpi=150)
    plt.show()
else:
    print('(Training was skipped — weights already existed)')


## 4.9  Load Trained Diffusion Model

Load the **EMA weights** of the trained DDPM denoiser.

In [ ]:
assert os.path.exists(DIFF_WEIGHTS_FILE), (
    f'Weights not found: {DIFF_WEIGHTS_FILE}\n'
    'Please run Section 4.7 first (or upload pre-trained weights to Drive).'
)

diffusion_model.load_state_dict(
    torch.load(DIFF_WEIGHTS_FILE, map_location=DEVICE, weights_only=True)
)
diffusion_model.eval()
print(f'✅ Loaded EMA weights from {DIFF_WEIGHTS_FILE}')


## 4.10  DiffPIR Inference Functions (from )

The DiffPIR logic is implemented in the **IPPy fork** at
 and exposed via .  The four
public functions are:

| Function | Description |
|----------|-------------|
|  | FFT of the avg-pool box-filter PSF (zero-shift, periodic wrap) |
|  | Closed-form HQS step in Fourier domain |
|  | DDIM reverse step with proximal x̂₀ |
|  | Full DiffPIR loop (all-in-one) |

### Mathematical background (HQS proximal step)

For SR with average-pooling operator $ (stride $), the proximal
subproblem has a closed-form Fourier-domain solution:

35854\hat{x}_{	ext{prox}} = \mathcal{F}^{-1}\!\left[rac{\mathcal{H}^*(u,v)\cdot\mathcal{F}(K^	op y)(u,v)\ +\ ho\,\mathcal{F}(\hat{x}_0)(u,v)}{|\mathcal{H}(u,v)|^2 + ho}ight]35854

The HQS parameter adapts with the diffusion step:

35854ho_t = \kappa \cdot rac{(2\sigma_n)^2}{1 - ar{lpha}_t}35854

At large $ (high noise) $ho_t 	o 0$ → prior dominates.  
At small $ (low noise) $ho_t$ grows → data fidelity dominates.

In [ ]:
# ── Precompute DDIM strided schedule ────────────────────────────────────
# Evenly spaced indices in [0, T-1] → reversed for inference
ddim_step_indices  = torch.linspace(0, T_DIFFUSION - 1, DDIM_STEPS,
                                    dtype=torch.long)
alpha_bars_ddim    = alpha_bars_sched[ddim_step_indices]  # ᾱ at DDIM steps

# ── Import DiffPIR functions from IPPy ───────────────────────────────────
from IPPy.nn.diffpir import (
    get_sr_psf_fft,       # Fourier response of avg-pool PSF
    hqs_proximal_sr,      # closed-form HQS proximal step
    ddim_update,          # DDIM reverse diffusion step
    diffpir_reconstruct,  # full DiffPIR reconstruction loop
)

print("✅ DiffPIR functions imported from IPPy.nn.diffpir")


## 4.11  Test Dataset with Raw LR Observations

DiffPIR needs the **raw LR observation** $y^\delta$ (not the bilinearly upsampled
version used by NB03). We define a dataset that returns both the LR image and the
HR ground truth.

In [ ]:
from IPPy.operators import DownScaling

class DiffPIRDataset(Dataset):
    """
    Returns (y_lr, x_true) pairs for DiffPIR evaluation:
      y_lr   : (1, H//s, W//s) noisy LR observation in [0,1]
      x_true : (1, H, W) HR ground truth in [0,1]
    """
    def __init__(self, data_dir: str, scale_factor: int, noise_level: float,
                 img_shape=(256, 256)):
        self.files  = sorted(glob.glob(os.path.join(data_dir, '*.pt')))
        assert len(self.files) > 0, f'No .pt files in {data_dir}'
        self.K          = DownScaling(img_shape=img_shape,
                                      downscale_factor=scale_factor, mode='avg')
        self.noise_level = noise_level

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        x_true  = torch.load(self.files[idx], weights_only=True)  # (1, H, W)
        x_batch = x_true.unsqueeze(0)                             # (1,1, H, W)
        y_clean = self.K(x_batch)                                 # (1,1,H//s,W//s)
        y_noisy = y_clean + gaussian_noise(y_clean, self.noise_level)
        y_lr    = y_noisy.squeeze(0).clamp(0.0, 1.0)             # (1,H//s,W//s)
        return y_lr, x_true

print('DiffPIRDataset defined ✅')


## 4.12  DiffPIR Evaluation — All Scenarios

For each degradation scenario $(s, \sigma)$ we run DiffPIR on the first
`NUM_TEST_IMAGES` images and record **PSNR** and **SSIM** against the HR ground truth.

> ⏱  Expected time on Colab T4: ~30–60 s per image × `NUM_TEST_IMAGES`.

In [ ]:
all_results = {}   # {(s, sigma): {'psnr': [...], 'ssim': [...], 'samples': [...]}}

for s, sigma in ALL_SCENARIOS:
    tag = f's{s}_sig{str(sigma).replace(".", "")}'
    print(f'\n{"="*60}')
    print(f'  Scenario: scale={s}×, σ={sigma}  [{tag}]')
    print(f'{"="*60}')

    # Precompute PSF FFT once per scale factor
    H_fft = get_sr_psf_fft(s, *IMG_SHAPE, device=DEVICE)

    test_ds  = DiffPIRDataset(os.path.join(DATASET_PATH, 'test'), s, sigma, IMG_SHAPE)
    n_eval   = min(NUM_TEST_IMAGES, len(test_ds))
    print(f'  Evaluating on {n_eval}/{len(test_ds)} test images ...')

    psnr_list, ssim_list, samples = [], [], []

    for idx in range(n_eval):
        y_lr, x_true = test_ds[idx]
        y_lr_b  = y_lr.unsqueeze(0)    # (1,1,H//s,W//s)
        x_true_b = x_true.unsqueeze(0) # (1,1,H,W)

        t_start = time.time()
        x_hat = diffpir_reconstruct(
            model             = diffusion_model,
            y_lr_01           = y_lr_b,
            scale_factor      = s,
            sigma_n           = sigma,
            alpha_bars_ddim   = alpha_bars_ddim.to(DEVICE),
            ddim_step_indices = ddim_step_indices,
            H_fft             = H_fft,
            img_shape         = IMG_SHAPE,
            rho_scale         = RHO_SCALE,
            eta               = ETA,
        )
        elapsed = time.time() - t_start

        # Bilinear baseline (for reference)
        y_up = F.interpolate(y_lr_b, size=IMG_SHAPE, mode='bilinear',
                             align_corners=False)

        psnr_val = metrics.PSNR(x_hat, x_true_b)
        ssim_val = metrics.SSIM(x_hat, x_true_b)
        psnr_bic = metrics.PSNR(y_up.cpu(), x_true_b)
        ssim_bic = metrics.SSIM(y_up.cpu(), x_true_b)

        psnr_list.append(psnr_val)
        ssim_list.append(ssim_val)
        if idx < 4:   # store first 4 for visualisation
            samples.append({'y_lr': y_lr_b, 'y_up': y_up, 'x_hat': x_hat,
                            'x_true': x_true_b})

        print(f'  [{idx+1:3d}/{n_eval}]  PSNR {psnr_val:.2f} dB  '
              f'(bic {psnr_bic:.2f})  |  SSIM {ssim_val:.4f}  '
              f'(bic {ssim_bic:.4f})  [{elapsed:.0f}s]')

    mean_psnr = float(np.mean(psnr_list))
    mean_ssim = float(np.mean(ssim_list))
    all_results[(s, sigma)] = {
        'psnr': psnr_list, 'ssim': ssim_list, 'samples': samples,
        'mean_psnr': mean_psnr, 'mean_ssim': mean_ssim,
    }
    print(f'\n  ➜  Mean PSNR = {mean_psnr:.2f} dB  |  Mean SSIM = {mean_ssim:.4f}')


## 4.13  Quantitative Results

In [ ]:
print(f'{'─'*55}')
print(f"{'Scenario':<22} | {'PSNR (dB)':>10} | {'SSIM':>8}")
print(f'{'─'*55}')
for (s, sigma), res in all_results.items():
    label = f'scale={s}×, σ={sigma}'
    print(f"{label:<22} | {res['mean_psnr']:>10.2f} | {res['mean_ssim']:>8.4f}")
print(f'{'─'*55}')
print(f'\n  Evaluated on {NUM_TEST_IMAGES} images per scenario, '
      f'DDIM_STEPS={DDIM_STEPS}, RHO_SCALE={RHO_SCALE}')


## 4.14  Qualitative Visualisation

For each scenario we show one example: Ground Truth | LR (bilinear) | DiffPIR reconstruction.

In [ ]:
def show_scenario(ax_row, sample, s, sigma):
    x_true = sample['x_true'][0, 0].cpu().numpy()
    y_up   = sample['y_up'][0, 0].cpu().numpy()
    x_hat  = sample['x_hat'][0, 0].cpu().numpy()

    psnr_v = metrics.PSNR(sample['x_hat'], sample['x_true'])
    ssim_v = metrics.SSIM(sample['x_hat'], sample['x_true'])
    psnr_b = metrics.PSNR(sample['y_up'].cpu(), sample['x_true'])
    ssim_b = metrics.SSIM(sample['y_up'].cpu(), sample['x_true'])

    for ax, img, title in zip(
        ax_row,
        [x_true, y_up, x_hat],
        ['Ground Truth',
         f'Bicubic Up\nPSNR {psnr_b:.2f} dB  SSIM {ssim_b:.4f}',
         f'DiffPIR\nPSNR {psnr_v:.2f} dB  SSIM {ssim_v:.4f}']
    ):
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title, fontsize=8)
        ax.axis('off')

    ax_row[0].set_ylabel(f's={s}×, σ={sigma}', fontsize=8, rotation=0,
                         labelpad=60, va='center')

n_rows = len(all_results)
fig, axes = plt.subplots(n_rows, 3, figsize=(10, 3.2 * n_rows))
if n_rows == 1:
    axes = [axes]

for ax_row, ((s, sigma), res) in zip(axes, all_results.items()):
    if res['samples']:
        show_scenario(ax_row, res['samples'][0], s, sigma)

plt.suptitle('DiffPIR — Qualitative Results', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_PATH, 'diffpir_qualitative.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {os.path.join(OUTPUT_PATH, "diffpir_qualitative.png")}')


## 4.15  Save Results

Results are saved in the same format as NB03 so that NB05 can aggregate them.

In [ ]:
import csv

# ── CSV with per-image metrics ───────────────────────────────────────────
csv_path = os.path.join(OUTPUT_PATH, 'diffpir_metrics.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['method', 'scale_factor', 'noise_level', 'image_idx',
                     'psnr_db', 'ssim'])
    for (s, sigma), res in all_results.items():
        for img_idx, (psnr_v, ssim_v) in enumerate(zip(res['psnr'], res['ssim'])):
            writer.writerow(['diffpir', s, sigma, img_idx, psnr_v, ssim_v])
print(f'Metrics CSV saved → {csv_path}')

# ── Save reconstructed tensors (.pt) for NB05 comparison ─────────────────
for (s, sigma), res in all_results.items():
    tag = f's{s}_sig{str(sigma).replace(".", "")}'
    pt_dir = os.path.join(OUTPUT_PATH, tag)
    os.makedirs(pt_dir, exist_ok=True)
    for img_idx, sample in enumerate(res['samples']):
        out_file = os.path.join(pt_dir, f'diffpir_{img_idx:04d}.pt')
        torch.save(sample['x_hat'], out_file)
print(f'Reconstructed tensors saved under {OUTPUT_PATH}/')

# ── Summary table ────────────────────────────────────────────────────────
print('\n=== DiffPIR Summary ===')
print(f"{'Scenario':<22} | {'Mean PSNR':>10} | {'Mean SSIM':>10}")
print('-' * 50)
for (s, sigma), res in all_results.items():
    print(f"  s={s}, σ={sigma}{'':10} | {res['mean_psnr']:>10.2f} | {res['mean_ssim']:>10.4f}")
